# Python Bridge for 07b

## A Short Code Refresher

Use this optional notebook alongside 07b for a closer look at the Python mechanics of hierarchical clustering. Plan for about 15–20 minutes. It reuses the lecture's six-point example and includes all the data and setup here.

The lecture explains how linkage rules build a hierarchy. This companion follows five code patterns:

1. supply a rows-by-features array to SciPy;
2. read the returned merge-history array;
3. pass that array to a dendrogram;
4. extract membership labels for a cut; and
5. attach the labels to the original observations.

## References

SciPy documents the inputs and outputs of [linkage](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html), [dendrogram](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.dendrogram.html), [cut_tree](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.cut_tree.html), and [fcluster](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.fcluster.html). The [Course Addenda and Errata page](https://auburn.instructure.com/courses/1747056/pages/course-addenda-and-errata) discusses the separate question of assigning new observations to fixed groups.

## Create the Six-Point Table

The two coordinates use the same units, so we use them directly. In the lecture's customer application, the inputs instead come from a fitted scaler. In both cases, each input row must describe one observation.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from scipy.cluster.hierarchy import cut_tree, dendrogram, fcluster, linkage

In [ ]:
points = pd.DataFrame(
    {"x": [0, 1, 3, 6, 10, 11], "y": [0, 0, 1, 3, 0, 1]},
    index=list("ABCDEF"),
)
values = points.to_numpy()

print("input shape:", values.shape)
points

The shape `(6, 2)` means six observations and two coordinates. `.to_numpy()` removes the pandas labels, so keep `points.index` for later displays.

## Build the Merge History

`linkage()` is a function that returns a numerical array. `method="complete"` measures separation between two groups by their largest cross-pair distance. `metric="euclidean"` specifies how the original point distances are calculated.

In [ ]:
tree = linkage(values, method="complete", metric="euclidean")
merge_table = pd.DataFrame(tree, columns=["Left ID", "Right ID", "Height", "Member count"])
merge_table.index = range(len(points), len(points) + len(tree))
merge_table.index.name = "New cluster ID"

print("merge-history shape:", tree.shape)
merge_table.round(3)

Six observations require five merges, giving a `(5, 4)` array. Original row positions 0–5 identify A–F. The first merge creates ID 6, the second creates ID 7, and row `i` creates ID `6 + i`. The first row joins IDs 0 and 1 at distance 1 and records two members. The second joins IDs 4 and 5, creating ID 7 for {E,F}.

SciPy stores all four columns as floating-point numbers because they share one array. The ID and member-count columns still represent whole numbers. These IDs describe nodes in the merge history; membership labels from a cut will be a different set of identifiers.

Pass the observation coordinates to this call. A square distance matrix would be interpreted as a feature table. If starting from pairwise distances instead, SciPy expects its one-dimensional condensed-distance format.

## Draw the Existing Tree

`dendrogram()` uses the merge history already calculated. It does not fit a second hierarchy. `labels` supplies names corresponding to the original input row positions.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
drawing = dendrogram(
    tree,
    labels=points.index.tolist(),
    color_threshold=0,
    above_threshold_color="#334155",
    ax=ax,
)
ax.axhline(4, color="#b45309", linestyle="--", label="Height 4")
ax.set(
    xlabel="Observation",
    ylabel="Complete-linkage distance",
    title="Six Observations, One Merge History",
)
ax.legend()
plt.show()

print("row positions in drawing order:", drawing["leaves"])

The cut at height 4 crosses three branches: {A,B,C}, {D}, and {E,F}. The leaf order can differ from the input row order. It arranges the drawing; it does not reorder the membership arrays returned by the cut functions.

## Request Three Clusters

`cut_tree()` returns one column per requested cut. With one count requested, the result still has two dimensions. `.ravel()` turns its single column into the one-dimensional label vector needed for a DataFrame column.

In [ ]:
cut_array = cut_tree(tree, n_clusters=3)
labels_three = cut_array.ravel()

print("cut-array shape:", cut_array.shape)
print("label-vector shape:", labels_three.shape)
points.assign(cluster=labels_three)

The shapes are `(6, 1)` and `(6,)`. Labels follow A–F, the original input order. Observations with equal labels share a cluster. A label of 0 here does not refer to original point ID 0 or to the first merged node.

## Compare Two Cuts of the Same Tree

A list of cluster counts requests several cuts at once. The first output column corresponds to two clusters and the second to three.

In [ ]:
cuts = cut_tree(tree, n_clusters=[2, 3])
cut_table = pd.DataFrame(cuts, index=points.index, columns=["Two", "Three"])

cut_table

The three-cluster partition splits {D,E,F} into {D} and {E,F}, while {A,B,C} remains intact. These are nested cuts of the same hierarchy. Changing the requested count does not rebuild the merge history.

## Cut by Height Instead

With `criterion="distance"`, `fcluster()` interprets `t` as a merge-height threshold. Its result is already a one-dimensional array.

In [ ]:
height_labels = fcluster(tree, t=4, criterion="distance")
comparison = points.assign(
    count_cut=labels_three,
    height_cut=height_labels,
)

comparison

The two columns give the same member sets with different numerical labels. Compare memberships, not whether the raw label arrays are identical. If you use `criterion="maxclust"`, `t` instead requests at most that many clusters; tied merge heights can yield fewer.

## Summarize the Assigned Rows

Attach the labels before grouping, preserving their connection to the observations. Selecting `points.columns` restricts the mean calculation to the original coordinates.

In [ ]:
labeled = points.assign(cluster=labels_three)
summary = labeled.groupby("cluster")[points.columns].mean()
summary["Members"] = labeled.groupby("cluster").size()

summary.round(3)

The member counts are 3, 1, and 2. The coordinate means describe the chosen groups. Complete linkage did not use those means to build the hierarchy.

## Try One Small Modification

Change `t=4` to `t=7` in the height-cut cell, then rerun that cell. D should join E and F, while A–C stay together. The `count_cut` column still describes the earlier three-cluster cut. You changed how the existing tree was cut, not the tree itself.

## Ready for 07b

You are ready to return to 07b when you can recognize these patterns:

```text
values = points.to_numpy()                   # retain row order, remove labels
linkage(values, method="complete")          # return the merge history
dendrogram(tree, labels=...)                 # draw that history
cut_tree(tree, n_clusters=3).ravel()         # extract a count-based cut
fcluster(tree, t=4, criterion="distance")    # extract a height-based cut
points.assign(cluster=labels)                # restore the row-label connection
```

`tree` is an array, not a fitted estimator with a `.predict()` method. Including another observation in `linkage()` builds a new hierarchy and can change earlier memberships. Assigning new rows while keeping existing groups fixed requires a separately chosen assignment rule, as explained in the addendum.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).